# Dataset SFT para fine-tuning Llama 3.2 3B

Este notebook transforma os PDFs extraídos em `fontes_saude_mulher_v2.json` em um dataset de **Supervised Fine-Tuning (SFT)** no formato chat do Llama 3.

1. **Chunking** 
2. **Geração de Q&A** 
3. **Tratamento de sensitive** 
4. **Formato Llama 3 chat** 
5. **Saída**: `sft_train.jsonl`, `sft_val.jsonl`, `sft_test.jsonl` em `files/sft/`


In [34]:
!pip install -q transformers accelerate bitsandbytes peft sentencepiece python-dotenv

In [35]:
import os
import json
import re
import torch
import random

from getpass import getpass
from huggingface_hub import login
from google.colab import drive
from dotenv import load_dotenv
from pathlib import Path
from collections import Counter, defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [36]:
drive.mount('/content/drive', force_remount=True)

DRIVE_FILES_DIR = '/content/drive/MyDrive/AssistenteHospitalar/files'
SOURCE_JSON = f'{DRIVE_FILES_DIR}/fontes_saude_mulher_v2.json'
OUTPUT_DIR = f'{DRIVE_FILES_DIR}/sft'

ENV_PATH = '/content/drive/MyDrive/AssistenteHospitalar/.env'
if not load_dotenv(ENV_PATH):
    raise FileNotFoundError(
        f'.env não encontrado em {ENV_PATH}. '
        'Crie o arquivo com a linha: HF_TOKEN=seu_token_aqui'
    )

HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError('HF_TOKEN não definido no .env')

login(token=HF_TOKEN)
print('HuggingFace autenticado.')

In [ ]:
# --- Configuração ---
GENERATOR_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'   # modelo que GERA os Q&A
TARGET_MODEL    = 'meta-llama/Llama-3.2-3B-Instruct'   # modelo-alvo do fine-tuning (referência)

CHUNK_CHAR_SIZE = 6000      # ~1500 tokens, deixa ~2K p/ prompt+resposta no contexto 8K
CHUNK_OVERLAP   = 400       # sobreposição para não cortar contexto importante
QA_PER_CHUNK    = 4         # quantos pares Q&A gerar por chunk
MAX_NEW_TOKENS  = 1024      # resposta do gerador

SAMPLE_PER_CATEGORY = None
SAMPLE_SEED = 42

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_SUFFIX = f'_n{SAMPLE_PER_CATEGORY}' if SAMPLE_PER_CATEGORY else '_full'
CHECKPOINT = OUTPUT_DIR / f'qa_pairs_raw{CHECKPOINT_SUFFIX}.jsonl'

print('Generator:', GENERATOR_MODEL)
print('Target:   ', TARGET_MODEL)
print('Output:   ', OUTPUT_DIR.resolve())
print(f'Amostragem: {SAMPLE_PER_CATEGORY or "TODOS"} chunks por categoria')
print(f'Checkpoint: {CHECKPOINT.name}')

In [38]:
with open(SOURCE_JSON, 'r', encoding='utf-8') as f:
    docs = json.load(f)

print(f'Documentos carregados: {len(docs)}')
print('\nDistribuição por categoria:')
by_cat = Counter(d['category'] for d in docs)
for cat, n in sorted(by_cat.items(), key=lambda x: -x[1]):
    sens = '(sensitive)' if any(d['sensitive'] and d['category']==cat for d in docs) else ''
    print(f'  {cat:<25} {n} docs {sens}')

print('\nTamanho de conteúdo (caracteres):')
lens = sorted([len(d['content']) for d in docs])
print(f'  min={lens[0]:,}  mediana={lens[len(lens)//2]:,}  max={lens[-1]:,}')

In [39]:
def chunk_text(text, size=CHUNK_CHAR_SIZE, overlap=CHUNK_OVERLAP):
    """Divide texto em chunks de ~size caracteres tentando quebrar em parágrafos."""
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    if len(text) <= size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        # tenta cortar em quebra de parágrafo
        if end < len(text):
            cut = text.rfind('\n\n', start, end)
            if cut > start + size // 2:
                end = cut
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = end - overlap
    return [c for c in chunks if len(c) > 200]

# Aplica chunking e mostra estatísticas
chunked = []
for doc in docs:
    for i, ch in enumerate(chunk_text(doc['content'])):
        chunked.append({
            'doc_id': doc['filename'],
            'chunk_id': f"{doc['filename']}::{i}",
            'category': doc['category'],
            'sensitive': doc['sensitive'],
            'name': doc['name'],
            'text': ch,
        })

print(f'Total de chunks: {len(chunked)}')
print('Chunks por categoria:')
by_cat_chunks = Counter(c['category'] for c in chunked)
for cat, n in sorted(by_cat_chunks.items(), key=lambda x: -x[1]):
    print(f'  {cat:<25} {n}')
print(f'\nEstimativa Q&A finais: {len(chunked) * QA_PER_CHUNK} pares')

In [40]:

if SAMPLE_PER_CATEGORY is not None:
    rng = random.Random(SAMPLE_SEED)
    by_cat = defaultdict(list)
    for c in chunked:
        by_cat[c['category']].append(c)

    sampled = []
    for cat, items in by_cat.items():
        rng.shuffle(items)
        take = min(SAMPLE_PER_CATEGORY, len(items))
        sampled.extend(items[:take])

    print(f'Amostragem ativa: máx {SAMPLE_PER_CATEGORY} chunks por categoria (seed={SAMPLE_SEED})')
    print(f'Chunks antes: {len(chunked)} -> depois: {len(sampled)}')
    chunked = sampled
else:
    print(f'Sem amostragem — usando todos os {len(chunked)} chunks')

print('\nDistribuição final:')
for cat, n in sorted(Counter(c['category'] for c in chunked).items(), key=lambda x: -x[1]):
    print(f'  {cat:<25} {n}')

print(f'\nTempo estimado: ~{len(chunked) * 20 / 60:.0f} min (a ~20s/chunk no A100)')
print(f'Estimativa Q&A finais: {len(chunked) * QA_PER_CHUNK} pares')

In [42]:
# Carrega Llama 3.1 8B Instruct em 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    dtype=torch.float16,
)
model.eval()
print('Modelo carregado. Device:', next(model.parameters()).device)

In [ ]:
# Prompts: voltados para PROFISSIONAIS DE SAÚDE do hospital (médicos, enfermeiros, técnicos),

PROMPT_PADRAO = """Você está preparando dados de treinamento para um assistente virtual usado pela EQUIPE DE SAÚDE de um hospital (médicos, enfermeiros, residentes, técnicos) durante o atendimento a mulheres.

A partir do TRECHO DE PROTOCOLO abaixo, gere {n} pares de PERGUNTA e RESPOSTA realistas, no formato JSON.

Regras OBRIGATÓRIAS:
- As perguntas devem ser do tipo que um profissional de saúde faria durante o atendimento (ex.: conduta indicada, posologia, critérios diagnósticos, fluxo de encaminhamento, critérios de internação, exames complementares, contraindicações, sinais de alarme).
- Linguagem TÉCNICA é apropriada — pode usar termos clínicos, siglas (CID, RN, IG, DUM, PA, FCF, LSIL, HSIL, etc.) e jargão médico quando aparecer no protocolo.
- As respostas devem ser BASEADAS ESTRITAMENTE no texto fornecido — não invente condutas, doses, critérios ou fluxos. Se o protocolo não responde claramente, não gere o par.
- Respostas devem ser OBJETIVAS, no tom de protocolo clínico: cite a conduta, dose, critério ou fluxo de forma direta e acionável. Sem rodeios, sem linguagem para leigos.
- Em português brasileiro.
- NÃO oriente o leitor a "procurar um profissional de saúde" — o leitor JÁ é o profissional.

Categoria do protocolo: {category}

TRECHO DE PROTOCOLO:
---
{text}
---

Responda APENAS com um JSON válido neste formato (sem texto antes ou depois):
{{"pairs": [{{"q": "...", "a": "..."}}, ...]}}"""


PROMPT_SENSITIVE = """Você está preparando dados de treinamento para um assistente virtual usado pela EQUIPE DE SAÚDE de um hospital ao atender mulheres em situações DELICADAS (violência doméstica, sexual, sofrimento mental, depressão pós-parto, risco de suicídio).

A partir do TRECHO DE PROTOCOLO abaixo, gere {n} pares de PERGUNTA e RESPOSTA, no formato JSON. As perguntas são feitas pelo PROFISSIONAL DE SAÚDE durante o atendimento — NUNCA pela paciente.

Regras OBRIGATÓRIAS para conteúdo sensível:
- As perguntas devem refletir dúvidas reais do profissional durante o atendimento. Exemplos de formatos válidos:
  - "Como conduzir a escuta qualificada de uma paciente com suspeita de violência doméstica?"
  - "Quais sinais de alerta para risco de suicídio devo identificar na anamnese?"
  - "Como preencher a ficha de notificação compulsória (SINAN) em caso de violência?"
  - "Qual o fluxo de profilaxia pós-exposição em caso de violência sexual?"
  - "Quais critérios indicam internação psiquiátrica involuntária?"
  - "Quando devo acionar a Delegacia da Mulher / Conselho Tutelar?"
- As respostas devem ser TÉCNICAS e ACIONÁVEIS para o profissional: cite o protocolo de acolhimento, a notificação obrigatória (SINAN para violência), os fluxos da rede, os critérios objetivos, os prazos legais.
- Quando o protocolo mencionar serviços da rede (Ligue 180, CVV 188, CAPS, SAMU 192, Delegacia da Mulher, Centro de Referência), inclua-os como **informação que o profissional deve passar à paciente** — não como direcionado ao próprio profissional.
- Linguagem técnica é apropriada. Mantenha o tom de protocolo clínico, NÃO o tom de acolhimento direto à vítima.
- BASEIE-SE ESTRITAMENTE no texto fornecido. Se o protocolo não responde claramente, não gere o par.
- NUNCA invente condutas, doses, prazos ou fluxos.
- Em português brasileiro.

Categoria do protocolo: {category}

TRECHO DE PROTOCOLO:
---
{text}
---

Responda APENAS com um JSON válido neste formato (sem texto antes ou depois):
{{"pairs": [{{"q": "...", "a": "..."}}, ...]}}"""

In [ ]:
def build_prompt(chunk):
    tpl = PROMPT_SENSITIVE if chunk['sensitive'] else PROMPT_PADRAO
    return tpl.format(n=QA_PER_CHUNK, category=chunk['category'], text=chunk['text'])

def generate_qa(chunk):
    prompt = build_prompt(chunk)
    messages = [
        {'role': 'system', 'content': 'Você é um médico especialista em saúde da mulher gerando dados de treinamento para um assistente clínico usado pela equipe de saúde de um hospital. Responda APENAS com JSON válido conforme solicitado.'},
        {'role': 'user', 'content': prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    ).to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
    return text


def _strip_fences(text):
    """Remove ```json ... ``` ou ``` ... ``` ao redor do JSON."""
    text = re.sub(r'^```(?:json)?\s*', '', text.strip(), flags=re.IGNORECASE)
    text = re.sub(r'\s*```$', '', text)
    return text.strip()


def _extract_pairs_regex(text):
    """Fallback: extrai pares q/a por regex mesmo com JSON malformado."""
    pattern = re.compile(
        r'"q"\s*:\s*"((?:[^"\\]|\\.)*)"\s*,\s*"a"\s*:\s*"((?:[^"\\]|\\.)*)"',
        re.DOTALL,
    )
    pairs = []
    for m in pattern.finditer(text):
        q = m.group(1).encode().decode('unicode_escape', errors='ignore').strip()
        a = m.group(2).encode().decode('unicode_escape', errors='ignore').strip()
        if q and a:
            pairs.append({'q': q, 'a': a})
    return pairs


def parse_qa_json(text):
    """Extrai pares Q&A do JSON gerado, com vários fallbacks."""
    cleaned = _strip_fences(text)

    # 1) tenta JSON direto (do primeiro { ao último } que contenha "pairs")
    m = re.search(r'\{[^{}]*"pairs"\s*:\s*\[.*\]\s*\}', cleaned, re.DOTALL)
    if not m:
        m = re.search(r'\{.*"pairs".*\}', cleaned, re.DOTALL)
    if m:
        candidate = m.group(0)
        # remove vírgulas finais (Llama costuma colocar)
        candidate_fixed = re.sub(r',\s*([\]}])', r'\1', candidate)
        for attempt in (candidate, candidate_fixed):
            try:
                obj = json.loads(attempt)
                pairs = [
                    {'q': p['q'].strip(), 'a': p['a'].strip()}
                    for p in obj.get('pairs', [])
                    if isinstance(p, dict) and 'q' in p and 'a' in p
                ]
                if pairs:
                    return pairs
            except json.JSONDecodeError:
                continue

    # 2) fallback: extrai pares por regex (resiliente a quebras de linha em strings)
    return _extract_pairs_regex(cleaned)

In [44]:
# Diagnóstico: gera 1 chunk e mostra saída crua + resultado do parser.
# Use isto antes de rodar o loop principal para confirmar que o parser está extraindo pares.
_debug_chunk = chunked[0]
print(f'Chunk de teste: {_debug_chunk["chunk_id"]} (categoria={_debug_chunk["category"]}, sensitive={_debug_chunk["sensitive"]})')
print('-' * 80)
_raw = generate_qa(_debug_chunk)
print('--- SAÍDA CRUA DO MODELO ---')
print(_raw[:2000])
print('...' if len(_raw) > 2000 else '')
print('-' * 80)
_pairs = parse_qa_json(_raw)
print(f'Pares extraídos: {len(_pairs)}')
for i, p in enumerate(_pairs[:2], 1):
    print(f'\n[{i}] Q: {p["q"][:200]}')
    print(f'    A: {p["a"][:300]}')

In [45]:
# Loop principal com salvamento incremental (permite retomar se a sessão cair)
done_chunk_ids = set()
if CHECKPOINT.exists():
    with open(CHECKPOINT, 'r', encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            done_chunk_ids.add(row['chunk_id'])
    print(f'Retomando: {len(done_chunk_ids)} chunks já processados.')

remaining = [c for c in chunked if c['chunk_id'] not in done_chunk_ids]
print(f'Chunks a processar agora: {len(remaining)}')

for idx, chunk in enumerate(remaining, 1):
    try:
        raw = generate_qa(chunk)
        pairs = parse_qa_json(raw)
    except Exception as e:
        print(f'  ✗ erro em {chunk["chunk_id"]}: {e}')
        pairs = []
    row = {
        'chunk_id': chunk['chunk_id'],
        'doc_id': chunk['doc_id'],
        'category': chunk['category'],
        'sensitive': chunk['sensitive'],
        'pairs': pairs,
        'raw_len': len(raw) if 'raw' in dir() else 0,
    }
    f_out.write(json.dumps(row, ensure_ascii=False) + '\n')
    f_out.flush()
    if idx % 5 == 0 or idx == len(remaining):
        print(f'  [{idx}/{len(remaining)}] {chunk["chunk_id"]} -> {len(pairs)} pares')

print('\nGeração concluída. ')

In [ ]:
# Converte para formato chat do Llama 3 e aplica filtros de qualidade
import random
random.seed(42)

SYSTEM_PROMPT = (
    'Você é um assistente clínico para a equipe de saúde de um hospital '
    '(médicos, enfermeiros, residentes, técnicos) especializado em saúde da mulher. '
    'Responda em português brasileiro, com tom técnico e objetivo, baseado em protocolos clínicos. '
    'Use linguagem técnica e siglas médicas quando apropriado. '
    'Suas respostas devem ser acionáveis: cite conduta, dose, critério ou fluxo de forma direta. '
    'NÃO oriente o usuário a "procurar um profissional de saúde" — ele já é o profissional. '
    'Em situações sensíveis (violência, sofrimento mental, crise), oriente o profissional sobre '
    'protocolo de acolhimento, notificação compulsória (SINAN), fluxos da rede e critérios objetivos. '
    'Os serviços da rede (Ligue 180, CVV 188, CAPS, SAMU 192, Delegacia da Mulher) são informações '
    'que o profissional deve passar à paciente, não direcionadas a ele.'
)

examples = []
with open(CHECKPOINT, 'r', encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        for p in row['pairs']:
            q, a = p.get('q', '').strip(), p.get('a', '').strip()
            # filtros mínimos de qualidade
            if len(q) < 10 or len(a) < 20:
                continue
            examples.append({
                'messages': [
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': q},
                    {'role': 'assistant', 'content': a},
                ],
                'category': row['category'],
                'sensitive': row['sensitive'],
                'source_doc': row['doc_id'],
            })

print(f'Total de exemplos válidos: {len(examples)}')
print('Distribuição:')
for cat, n in sorted(Counter(e['category'] for e in examples).items(), key=lambda x: -x[1]):
    print(f'  {cat:<25} {n}')

In [47]:
# Split estratificado por categoria (80/10/10)
by_cat = defaultdict(list)
for ex in examples:
    by_cat[ex['category']].append(ex)

train, val, test = [], [], []
for cat, items in by_cat.items():
    random.shuffle(items)
    n = len(items)
    n_val = max(1, n // 10)
    n_test = max(1, n // 10)
    val   += items[:n_val]
    test  += items[n_val:n_val + n_test]
    train += items[n_val + n_test:]

random.shuffle(train); random.shuffle(val); random.shuffle(test)

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

save_jsonl(train, OUTPUT_DIR / 'sft_train.jsonl')
save_jsonl(val,   OUTPUT_DIR / 'sft_val.jsonl')
save_jsonl(test,  OUTPUT_DIR / 'sft_test.jsonl')

print(f'Train: {len(train)}  Val: {len(val)}  Test: {len(test)}')
print(f'\nArquivos salvos em: {OUTPUT_DIR.resolve()}')
print('  - sft_train.jsonl')
print('  - sft_val.jsonl')
print('  - sft_test.jsonl')

## ✅ Dataset gerado 